# LAB 7: 코드 확인

## 코드 확인 준비하기

시작하기에 앞서, 이 실습 전반에서 사용할 **컨텍스트 정보**를 가져오겠습니다. 

- **Start** 버튼을 클릭하여 이 노트북을 활성화하세요.

- 다음 Python 셀을 실행하세요.

#### :warning: 이 노트북에 대해 새 세션이 시작될 때마다, 후속 셀에서 사용할 '변수'를 구성하기 위해 아래 셀을 다시 실행해야 합니다. :warning:

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
your_db = user + '_DB'
print('현재 CONTEXT 정보:')
print('---------------------------------')
print(session)
print('현재 USER는 ' + user)

### INFORMATION_SCHEMA를 사용하여 메타데이터 쿼리하기 🥋

메타데이터란 '데이터에 대한 데이터'를 의미합니다. 

모든 Snowflake 데이터베이스에는 메타데이터를 보관하는 `INFORMATION_SCHEMA`가 생성됩니다. 즉, 이 스키마에는 데이터베이스, 스키마, 테이블, 뷰 등의 개수에 대한 통계 정보가 저장됩니다. 또한 오브젝트 이름과 기타 오브젝트 세부 정보에 대한 데이터도 포함되어 있습니다. 

우리는 `INFORMATION_SCHEMA`를 사용하여 작업 내용을 다시 한번 확인하고, 작업을 올바르게 완료했는지 검증할 수 있습니다. 

지금까지 작업해 온 **(animal)_GARDEN_PLANTS** 데이터베이스에 존재하는 모든 스키마를 확인하기 위해, 해당 데이터베이스의 `INFORMATION_SCHEMA`를 쿼리해 보겠습니다.

In [ ]:
%%sql -r dataframe_1
SELECT * 
FROM {{user}}_garden_plants.information_schema.schemata;

### 코드를 사용하여 작업 결과를 확인하세요. 🥋 

여러분은 **(animal)_GARDEN_PLANTS** 데이터베이스에 3개의 스키마를 생성해야 했고, 또한 하나의 스키마를 삭제해야 했습니다. 이제 해당 작업들이 제대로 완료되었는지 확인하기 위해 코드를 실행해 보겠습니다.

💡 **팁**: 다음 검사에 포함된 코드는 [공통 테이블 식(CTE)](https://docs.snowflake.com/ko/user-guide/queries-cte) 구조를 사용합니다. 공통 테이블 식(CTE)은 하나의 구문에서 사용할 수 있는 '임시 뷰’라고 생각할 수 있습니다. 공통 테이블 식(CTE)은 복잡한 SQL 구문을 더 작은 단위로 나누는 데 특히 유용하여, 가독성과 관리 용이성을 높여줍니다. 로직을 명확하고 재사용 가능한 부분으로 구성함으로써, 공통 테이블 식(CTE)은 쿼리 구조를 단순화하고 이해도를 높이며 유지보수성을 향상시킵니다.

In [ ]:
%%sql -r dataframe_2
WITH schema_check_1 AS (
    -- 다음 세 개의 스키마가 존재하는가?
    SELECT COUNT(*) AS count_1
    FROM {{user}}_garden_plants.information_schema.schemata
    WHERE schema_name IN ('FLOWERS','VEGGIES','FRUITS')
),
schema_check_2 AS (   
    -- 다음 스키마는 존재하지 않아야 합니다 (개수 0)
    SELECT COUNT(*) AS count_2
    FROM {{user}}_garden_plants.information_schema.schemata
    WHERE schema_name = ('PUBLIC')
) 
SELECT IFF((count_1=3) AND (count_2=0),'\u2705 Correct','\u26D4 Incorrect. Please review and try again') AS schema_check
from schema_check_1, schema_check_2;

## `(animal)_GARDEN_PLANTS` 데이터베이스에 스키마가 몇 개 있나요? 

### 무엇이 잘못되었나요? 📓 

위 쿼리를 실행했는데 오류가 발생했나요? 다음은 몇 가지 일반적인 실수입니다.

- 스키마 이름에 오타가 있습니다(예: 'VEGGIES' 대신 'WEGGIES').

- 스키마를 잘못된 데이터베이스에 넣었습니다(예: (user)_GARDEN_PLANTS 대신 UTIL_DB).

- 생성한 오브젝트를 볼 수 있도록 역할이 설정되지 않았습니다. 예를 들어, 오브젝트는 `(animal)_LEARNER_RL`로 생성했지만 현재 역할은 `PUBLIC`으로 설정되어 있습니다. 

---

### 해결 방법 📓 

**오타**: `ALTER SCHEMA (animal)_GARDEN_PLANTS.WEGGIES RENAME TO (animal)_GARDEN_PLANTS.VEGGIES;`

**잘못된 위치**: `ALTER SCHEMA DEMO_DB.VEGGIES RENAME TO (animal)_GARDEN_PLANTS.VEGGIES;`

**찾을 수 없음**: 역할 설정을 변경하거나 오브젝트의 소유권을 이전하세요. 

### 이름을 기준으로 스키마를 확인합니다. 🥋 

이제 생성한 스키마들이 올바른 이름을 가지고 있는지 확인해 보겠습니다. 만약 이 코드가 **3**개의 로우를 반환하지 않는다면, 스키마 이름을 다르게 지정했을 가능성이 있습니다. 

In [ ]:
%%sql -r dataframe_3
SELECT schema_name 
FROM {{user}}_GARDEN_PLANTS.INFORMATION_SCHEMA.SCHEMATA
WHERE schema_name IN ('FLOWERS','FRUITS','VEGGIES');

## 작업 결과 확인 🔎

### :mag_right: Check 1 (OB01) 🔎

- 3개의 Garden Plant 데이터베이스 스키마(**FLOWERS**, **VEGGIES**, **FRUITS**)를 생성했나요?
- 작업 결과 확인을 위해 채점용 Stored Procedure를 호출하세요.

In [ ]:
%%sql -r dataframe_4
CALL common_db.resources.local_grader('OB01', '{{user}}');

### :mag_right: Check 2 (OB02) 🔎

- Garden Plant 데이터베이스의 **PUBLIC**이라는 이름의 스키마를 삭제(drop)했나요?
- 작업 결과 확인을 위해 채점용 Stored Procedure를 호출하세요.

In [ ]:
%%sql -r dataframe_5
CALL common_db.resources.local_grader('OB02', '{{user}}');

### :mag_right: Check 3 (OB03) 🔎

- **ROOT_DEPTH** 테이블을 Garden Plant 데이터베이스의 **VEGGIES** 스키마에 생성했나요?
- 작업 결과 확인을 위해 채점용 Stored Procedure를 호출하세요.

In [ ]:
%%sql -r dataframe_6
CALL common_db.resources.local_grader('OB03', '{{user}}');

### Query History를 사용하여 테스트 결과를 검토하세요. 📓

Snowflake는 시스템에서 실행된 쿼리와 구문에 대한 레코드를 Query History라는 이름으로 보관하며, 이를 액세스할 수 있는 UI 방식과 프로그래밍 방식 모두를 제공합니다. Query History는 사용자가(또는 권한이 있는 경우 다른 사용자가) 그동안 실행한 쿼리들을 한 눈에 확인할 수 있는 편리한 공간입니다.

Snowflake Notebook의 셀을 통해서도 Query History에 액세스할 수 있습니다.

### Snowflake Notebook의 SQL 셀에서 Query History에 액세스해 보세요. 🥋

1. 방금 실행한 **Check 3** SQL 셀의 쿼리 실행 시간 표시 위에 마우스를 올리세요.
1. **View run details** 메시지가 나타납니다.
1. 쿼리 실행 시간 표시를 클릭하면 대화 상자가 나타납니다.
1. Snowsight Query History 페이지로 연결되는 링크가 포함된 파란색 **ID** 필드의 UUID를 클릭하세요.
1. 새 브라우저 창에서 Snowsight Query History 페이지가 열립니다.

![쿼리 기록 액세스(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_query_history_1_v2.png)

1. **Query Profile** 화면이 열리면 쿼리 실행 단계를 검토할 수 있습니다.
1. 화면 상단의 **Query Details** 탭을 클릭하세요.
1. 쿼리 실행과 관련된 다양한 세부 정보를 검토하세요.
1. 페이지 하단의 **Results** 섹션을 확인하세요. 

![쿼리 세부 정보(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_query_details_1_v2.png)

:warning: Query Details(Query History)의 Results 섹션에서 '확인 3' 항목에 녹색 체크 표시 ✅가 확인되지 않는 한, 이 단계 이후로 **진행하지 마세요**. :warning:

### 프로그래밍 방식으로 Query History에 액세스합니다. 🥋

Query History 정보를 코드를 통해 조회할 수도 있으며, Python과 SQL 모두에서 사용할 수 있는 옵션이 제공됩니다. 

`INFORMATION_SCHEMA`는 다양한 관점에서 Snowflake Query History를 조회하는 데 사용할 수 있는 [테이블 함수](https://docs.snowflake.com/ko/sql-reference/functions/query_history)들의 집합을 포함하고 있습니다. 다음 예시에서는 `QUERY_HISTORY_BY_USER()`를 사용하여, 최근 7일 이내에 특정 사용자(본인)가 실행한 쿼리들을 조회합니다. 

우리가 식별할 `DELETE` 작업은 **LAB 6: Load Wizard 및 Snowflake Marketplace**에서 실행했으며, **VEGETABLE_DETAILS** 테이블에서 단일 로우(… plant_name = 'Spinach' AND root_depth_code = 'D')을 삭제한 작업입니다.

In [ ]:
%%sql -r dataframe_7
SELECT *
FROM TABLE(information_schema.query_history_by_user(
    USER_NAME => '{{user}}',
    RESULT_LIMIT => 10000
))
WHERE query_type = 'DELETE'
AND execution_status = 'SUCCESS'
ORDER BY end_time DESC
LIMIT 1;

이 작업의 쿼리 ID를 SQL 변수에 저장하여 이 실습 후반부에 활용하겠습니다.

**:warning: `INFORMATION_SCHEMA.QUERY_HISTORY_BY_USER` 및 그 변형들은 데이터를 7일간만 보관합니다. 이 기간이 지난 후에 위 쿼리를 실행하면 결과가 반환되지 않습니다. 이 경우 LAB 6의 단계를 다시 수행해야 합니다.  :warning:**

In [ ]:
%%sql -r dataframe_8
SET delete_query_id = (
    SELECT query_id
    FROM TABLE(information_schema.query_history_by_user(
        USER_NAME => '{{user}}',
        RESULT_LIMIT => 10000
    ))
    WHERE query_type = 'DELETE'
    AND execution_status = 'SUCCESS'
    ORDER BY end_time DESC
    LIMIT 1
);

SELECT $delete_query_id;

### :mag_right: Check 4 (OB04) 🔎

- 데이터베이스 **(animal)_UTIL_DB**에 스키마가 2개(정확히 2개) 포함되어 있나요?
- 작업 결과 확인을 위해 채점용 Stored Procedure를 호출하세요.

In [ ]:
%%sql -r dataframe_9
CALL common_db.resources.local_grader('OB04', '{{user}}');

### :mag_right: Check 5 (OB05) 🔎

- **VEGETABLE_DETAILS** 테이블을 Garden Plant 데이터베이스의 **VEGGIES** 스키마에 생성했나요?
- 작업 결과 확인을 위해 채점용 Stored Procedure를 호출하세요.

In [ ]:
%%sql -r dataframe_10
CALL common_db.resources.local_grader('OB05', '{{user}}');

### :mag_right: Check 6 (OB06) 🔎

- **ROOT_DEPTH** 테이블에 **3**개의 로우가 있나요?
- 작업 결과 확인을 위해 채점용 Stored Procedure를 호출하세요.

In [ ]:
%%sql -r dataframe_11
CALL common_db.resources.local_grader('OB06', '{{user}}');  

### :mag_right: Check 7 (OB07) 🔎

- **VEGETABLE_DETAILS** 테이블에 **41**개의 로우가 있나요?
- 작업 결과 확인을 위해 채점용 Stored Procedure를 호출하세요.

In [ ]:
%%sql -r dataframe_12
CALL common_db.resources.local_grader('OB07', '{{user}}'); 

## Time Travel 📓

Snowflake Time Travel을 사용하면 정의된 기간 내 어느 시점에서든 변경되었거나 삭제된 과거 데이터에 액세스할 수 있습니다. Time Travel을 지원하기 위해 여러 [SQL 확장 기능](https://docs.snowflake.com/ko/user-guide/data-time-travel#time-travel-sql-extensions)이 제공됩니다.

데이터 삭제나 데이터가 포함된 오브젝트의 Drop을 포함하여 테이블에 있는 데이터를 수정하면 Snowflake는 업데이트 전의 데이터 상태를 보존합니다. `DATA_RETENTION_TIME_IN_DAYS`라는 파라미터는 이 과거 데이터가 보존되는 일 수를 지정하며, 그 결과 해당 기간 동안 데이터에 대해 Time Travel 작업(`SELECT`, `CREATE` … `CLONE`, `UNDROP`)을 수행할 수 있습니다.

### **VEGETABLE_DETAILS** 테이블의 과거 버전을 확인해 보세요. 🥋

앞서 **VEGETABLE_DETAILS** 테이블이 **LAB 6: Load Wizard 및 Snowflake Marketplace**에서 생성되었을 때, 7일간의 데이터 보존 기간으로 구성되었습니다. 이는 테이블에 대한 모든 변경 사항이 7일간의 기간 동안 보존됨을 의미합니다. 이를 통해 시간으로 거슬러 올라가 시간순 기준의 특정 시점이나 테이블에 대한 특정 작업이 실행되기 이전 시점의 데이터를 확인할 수 있습니다.

**LAB 6**에서 이후 두 번째 Spinach 로우를 **VEGETABLE_DETAILS** 테이블에서 삭제했습니다. 이 실습 초반에 해당 작업과 연관된 쿼리 ID를 확인하여 로컬 변수 `$delete_query_id`에 저장했습니다. Time Travel을 사용하면 이 작업 이전의 데이터를 확인할 수 있습니다. 

다음 코드를 실행하여 `DELETE` 이전의 Spinach 행들과 현재 버전의 테이블에 있는 Spinach 데이터를 **UNION**하여 확인하고 레이블을 지정하세요. Time Travel을 사용해 과거 데이터를 조회할 때 사용하는 [특수 구문](https://docs.snowflake.com/ko/user-guide/data-time-travel#querying-historical-data)인 `BEFORE( STATEMENT => $delete_query_id )`에 주목하세요. 

In [ ]:
%%sql -r dataframe_13
-- Time Travel 쿼리
SELECT plant_name, root_depth_code, 'HISTORICAL (Time Travel)' as table_version 
FROM {{user}}_garden_plants.veggies.vegetable_details
BEFORE( STATEMENT => $delete_query_id )
WHERE plant_name = 'Spinach'

UNION

-- 현재 버전 쿼리
SELECT plant_name, root_depth_code, 'CURRENT' as table_version
FROM {{user}}_garden_plants.veggies.vegetable_details
WHERE plant_name = 'Spinach';

## 지식 테스트 :mag_right:

아래의 대화형 퀴즈 문제를 통해 이해도를 확인해 보세요. 각 `RUN_THIS_QUIZ_QUESTION_` 셀에는 Snowflake 기능과 관련된 객관식 문제를 제시하는 Streamlit 위젯이 포함되어 있습니다.  

**지침:**  
1. 노트북 셀 위에 커서를 올려 추가 컨트롤을 표시하세요.
1. 각 퀴즈 셀 오른쪽의 ▶️ **Play 버튼**을 클릭하여 실행하세요.  
1. 제공된 옵션에서 답을 선택하세요.  
1. 다음으로 넘어가기 전에 피드백을 검토하세요. 

💡 **참고:** 궁금하시면 셀을 확장하여 코드를 볼 수 있지만, 필수는 아닙니다. 이 퀴즈들은 필수 사항이 아닙니다. 배운 내용을 복습하며 연습할 기회를 제공하기 위한 것입니다.  

In [ ]:
import streamlit as st
st.divider()
question = "메타데이터란 무엇인가요?"
options = ["아래 선택을 고르세요...",
           "A) 메타에 대한 데이터", 
           "B) 다른 데이터 위에 있는 데이터", 
           "C) 데이터에 대한 데이터"]

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "아래 선택을 고르세요...":
        ''
    else:
        answer = 'bf41cf3b90d11f3e3b826ba4c1f5eb91'
        # 옵션 문자 (A, B, C, 또는 D)를 추출하세요
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

In [ ]:
st.divider()
question = "Snowflake는 일부 메타데이터를 어디에 저장합니까?"
options = ["아래 선택을 고르세요...",
           "A) 각 계정의 INFORMATION_DB 데이터베이스", 
           "B) 각 데이터베이스의 METADATA_SCHEMA", 
           "C) 각 데이터베이스의 INFO_METADATA 스키마",
           "D) 각 데이터베이스의 INFORMATION_SCHEMA 스키마"]

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "아래 선택을 고르세요...":
        ''
    else:
        answer = 'e6e00de54b5311771ed1d02b87c85aa2'
        # 옵션 문자 (A, B, C, 또는 D)를 추출하세요
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

In [ ]:
st.divider()
question = "우리가 생성한 스키마를 확인하고 싶다면, 왜 직접 눈으로 확인하지 않을까요?"
options = ["아래 선택을 고르세요...",
           "A) 우리가 뭔가를 잘못 입력했을 경우, 이름 문제를 알아차리지 못할 수도 있지만, 코드를 통해 확인하면 그것을 잡아낼 수 있기 때문에", 
           "B) 어떤 이유로든 확인을 자동화하고 싶을 수 있기 때문에", 
           "C) 수동 확인이 코딩보다 훨씬 재미있기 때문에"]

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "아래 선택을 고르세요...":
        ''
    else:
        answer = '3a50be623ba4ad45ce1a260c91dface4'
        # 옵션 문자 (A, B, C, 또는 D)를 추출하세요
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

## 다음 단계

채점용 Check(1~7)을 모두 통과하고, **지식 확인** 질문에 올바르게 답했다면, Snowflake 강사의 안내에 따라 다음 Notebook으로 진행해 주세요.